# Q Discovery Check

This notebook uses the packaged `discover_q_from_processed_npy(...)` function from `CheckQ_SINDy` to validate Q-equation recovery on two processed datasets copied into this repository:

- `ANS`
- `ludwig_checkpoint_dense`


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

validator_root = Path(r"D:/Document/GitHub/ActiveBE_Validator")
checkq_root = Path(r"D:/Document/GitHub/CheckQ_SINDy")
checkq_src = checkq_root / "src"

if str(checkq_root) not in sys.path:
    sys.path.insert(0, str(checkq_root))
if str(checkq_src) not in sys.path:
    sys.path.insert(0, str(checkq_src))

from app import discover_q_from_processed_npy

datasets = [
    {
        "name": "ans",
        "q_path": validator_root / "example" / "data" / "ans" / "cache_q_stack.npy",
        "u_path": validator_root / "example" / "data" / "ans" / "cache_u_stack.npy",
    },
    {
        "name": "ludwig_checkpoint_dense",
        "q_path": validator_root / "example" / "data" / "ludwig_checkpoint_dense" / "cache_q_stack.npy",
        "u_path": validator_root / "example" / "data" / "ludwig_checkpoint_dense" / "cache_u_stack.npy",
    },
]

for dataset in datasets:
    print(dataset["name"], dataset["q_path"].exists(), dataset["u_path"].exists())


In [ ]:
results = []

for dataset in datasets:
    result = discover_q_from_processed_npy(
        q_npy_path=dataset["q_path"],
        velocity_npy_path=dataset["u_path"],
        dt=1.0,
        spacings=(1.0, 1.0, 1.0),
        q_time_discretization="two_point",
        v_time_level="n+1",
        q_time_level="n",
        q_sample_step=8,
        gamma=2.0,
        is_verbose=False,
    )
    results.append(
        {
            "name": dataset["name"],
            "r2": result.q_result.r2,
            "relative_residual": result.q_result.relative_residual,
            "params": result.inferred_q_parameters,
            "coefficients": dict(zip(result.q_result.feature_names, result.q_result.coefficients)),
        }
    )

for row in results:
    print(f"=== {row['name']} ===")
    print("Q r2                =", row["r2"])
    print("Q relative_residual =", row["relative_residual"])
    print("inferred params     =", row["params"])
    print("coefficients        =", row["coefficients"])
    print()


## Stored Reference Results

Recorded on August 7, 2026 using `discover_q_from_processed_npy(...)`:

- `ans`
  - `Q r2 = 1.0`
  - `Q relative_residual = 4.149245124746974e-14`
  - `lambda_r = 0.9999999999999949`
  - `lambda_1 = 0.666666666666664`
  - `lambda_2 = 0.9999999999999948`
  - `lambda_3 = -2.0000000000000124`

- `ludwig_checkpoint_dense`
  - `Q r2 = 0.9999074657343845`
  - `Q relative_residual = 0.009614481554594562`
  - `lambda_r = 1.0008890214524326`
  - `lambda_1 = 0.6662766030274542`
  - `lambda_2 = 1.0045474006248962`
  - `lambda_3 = -2.002953801430402`
